# 🚀 Notebook 05 — Gradio Deployment

Run this notebook **after** Notebook 04 to understand and test the Gradio app
before launching `app.py`.

## What this notebook does
```
Step 1  — Install Gradio
Step 2  — Load config + initialise all LangChain components  (same as app.py)
Step 3  — Define tools + agent                               (same as app.py)
Step 4  — Define get_video_suggestions() + build_thumbnail_html()
Step 5  — Test thumbnail generation in the notebook
Step 6  — Define the full chat_with_agent() function
Step 7  — Test a single question end-to-end
Step 8  — Launch Gradio inside the notebook  (inline preview)
Step 9  — How to switch to app.py for production
```

> **Requires:** `.env` with `OPENAI_API_KEY`, `PINECONE_API_KEY`, `LANGSMITH_API_KEY`
> and `config/agent_config.json` produced by Notebook 04.


## Step 1 — Install Gradio

In [3]:
# Run once if Gradio is not installed
!pip install gradio>=4.44.0
import gradio as gr
print(f"✅ Gradio {gr.__version__}")



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Gradio 6.9.0


## Step 2 — Imports & Configuration

Identical to `app.py` — we load the same `agent_config.json` produced by NB04.

In [4]:
import os, json, re
from typing import List, Tuple
from datetime import datetime
from dotenv import load_dotenv

from langchain.schema import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.tools import StructuredTool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_openai_functions_agent, AgentExecutor
from pydantic import BaseModel, Field

load_dotenv()

# ── Load config ────────────────────────────────────────────────────
config_path = "../config/agent_config.json"
with open(config_path) as f:
    config = json.load(f)

OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY  = os.getenv('PINECONE_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')

CHAT_MODEL      = config.get('chat_model',      'gpt-4o-mini')
EMBEDDING_MODEL = config.get('embedding_model', 'text-embedding-3-small')
INDEX_NAME      = config.get('index_name',      'youtube-rag-mechanical-engineering')
NAMESPACE       = config.get('namespace',       'efficient-engineer-v3')
TOP_K           = config.get('top_k',           5)

if LANGSMITH_API_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = config.get("langsmith_project", "youtube-rag-chatbot")
    os.environ["LANGCHAIN_API_KEY"]    = LANGSMITH_API_KEY

# ── Init components ────────────────────────────────────────────────
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)

print(f"✅ Config loaded: {CHAT_MODEL} | {INDEX_NAME}")


✅ Config loaded: gpt-4o-mini | youtube-rag-mechanical-engineering


## Step 3 — Tools & Agent

Same 3 tools as NB04. Copied here so this notebook is self-contained.

In [5]:
class SearchTranscriptsInput(BaseModel):
    query: str = Field(description="The search query to find relevant transcript content")

class GetVideoInfoInput(BaseModel):
    video_id: str = Field(description="The YouTube video ID to get information about")

class FindVideosInput(BaseModel):
    topic: str = Field(description="The topic to search for videos about")


def search_transcripts(query: str) -> str:
    """Search video transcripts for relevant content."""
    try:
        results = vectorstore.similarity_search(query, k=TOP_K)
        if not results:
            return "No relevant information found."
        formatted = []
        for i, doc in enumerate(results, 1):
            m     = doc.metadata
            vid   = m.get('video_id',    m.get('videoId',  'unknown'))
            title = m.get('video_title', m.get('title',    'Unknown Video'))
            t     = m.get('start_time',  m.get('start',    0))
            formatted.append(
                f"[Result {i}]\nVideo: {title}\nTime: {t}s\n"
                f"Content: {doc.page_content}\n"
                f"Link: https://youtube.com/watch?v={vid}&t={int(t)}s\n"
            )
        return "\n".join(formatted)
    except Exception as e:
        return f"Error: {e}"


def get_video_info(video_id: str) -> str:
    """Get metadata about a specific video."""
    try:
        results = vectorstore.similarity_search("", k=1, filter={"video_id": video_id})
        if not results:
            return f"Video {video_id} not found."
        m = results[0].metadata
        return (
            f"Title: {m.get('video_title', m.get('title', 'Unknown'))}\n"
            f"Channel: {m.get('channel', 'Unknown')}\n"
            f"Link: https://youtube.com/watch?v={video_id}"
        )
    except Exception as e:
        return f"Error: {e}"


def find_videos(topic: str) -> str:
    """Find videos related to a topic."""
    try:
        results = vectorstore.similarity_search(topic, k=TOP_K * 2)
        if not results:
            return "No videos found."
        seen, videos = set(), []
        for doc in results:
            vid = doc.metadata.get('video_id', doc.metadata.get('videoId'))
            if vid and vid not in seen:
                seen.add(vid)
                videos.append({
                    'title':    doc.metadata.get('video_title', doc.metadata.get('title', 'Unknown')),
                    'channel':  doc.metadata.get('channel', 'Unknown'),
                    'video_id': vid,
                })
            if len(videos) >= TOP_K: break
        return "\n\n".join(
            f"{i}. {v['title']}\n   Channel: {v['channel']}\n"
            f"   Link: https://youtube.com/watch?v={v['video_id']}"
            for i, v in enumerate(videos, 1)
        )
    except Exception as e:
        return f"Error: {e}"


tools = [
    StructuredTool.from_function(
        func=search_transcripts, name='search_transcripts',
        description=(
            "Search YouTube video transcripts for information. Use this when the user asks "
            "questions about engineering concepts, definitions, or explanations."
        ),
        args_schema=SearchTranscriptsInput,
    ),
    StructuredTool.from_function(
        func=find_videos, name='find_videos',
        description=(
            "Find videos about a specific topic. Use this when the user asks 'which videos', "
            "'show me videos', or wants to browse content about a subject."
        ),
        args_schema=FindVideosInput,
    ),
    StructuredTool.from_function(
        func=get_video_info, name='get_video_info',
        description="Get metadata about a specific video by its ID.",
        args_schema=GetVideoInfoInput,
    ),
]

system_message = """You are an expert mechanical engineering assistant with access to YouTube video transcripts.

IMPORTANT: Base your answers ONLY on the information found in the transcript search results.
Do not use outside knowledge. Always call search_transcripts before answering any question.

Current date: {current_date}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent          = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools,
    verbose=True, handle_parsing_errors=True, max_iterations=5,
)
print(f"✅ {len(tools)} tools + agent_executor ready")


✅ 3 tools + agent_executor ready


## Step 4 — Video Suggestion Helpers

These two functions are the **new feature** in the Gradio app:

- `get_video_suggestions(query, n)` — runs a Pinecone similarity search and returns
  a list of unique videos with their **thumbnail URL** and the **exact timestamp** of
  the most relevant chunk.
- `build_thumbnail_html(videos)` — turns that list into a row of clickable HTML cards.
  Each card shows the thumbnail, title, channel, and a timestamp badge.
  Clicking takes the user directly to that moment in the video.


In [6]:
def get_video_suggestions(query: str, n: int = 4) -> List[dict]:
    """
    Similarity-search Pinecone for `query`, return up to n unique videos.
    Each dict has: video_id, title, channel, start_time, thumbnail_url, watch_url
    watch_url already has the &t= timestamp so clicking jumps to the right spot.
    """
    try:
        results = vectorstore.similarity_search(query, k=n * 3)
    except Exception:
        return []

    seen, videos = set(), []
    for doc in results:
        m     = doc.metadata
        vid   = m.get('video_id',    m.get('videoId'))
        title = m.get('video_title', m.get('title',   'Unknown'))
        ch    = m.get('channel',     'Unknown')
        t     = m.get('start_time',  m.get('start',   0))
        # Use stored thumbnail or fall back to YouTube's CDN
        thumb = m.get("thumbnail", f"https://img.youtube.com/vi/{vid}/hqdefault.jpg")

        if vid and vid not in seen:
            seen.add(vid)
            videos.append({
                "video_id":      vid,
                "title":         title,
                "channel":       ch,
                "start_time":    int(t),
                "thumbnail_url": thumb,
                "watch_url":     f"https://www.youtube.com/watch?v={vid}&t={int(t)}s",
            })
        if len(videos) >= n:
            break
    return videos


def build_thumbnail_html(videos: List[dict]) -> str:
    """Render clickable video cards as an HTML string for gr.HTML()."""
    if not videos:
        return ""

    cards = []
    for v in videos:
        mins  = v['start_time'] // 60
        secs  = v['start_time'] %  60
        ts    = f"{mins}:{secs:02d}"
        title = v['title'][:52] + '…' if len(v['title']) > 52 else v['title']
        ch    = v['channel'][:30] + '…' if len(v['channel']) > 30 else v['channel']

        cards.append(f"""
        <a href="{v['watch_url']}" target="_blank" rel="noopener" style="
            text-decoration:none; color:inherit; display:block; width:210px; flex-shrink:0;
            border-radius:12px; overflow:hidden;
            background:#1a1a2e; border:1px solid #2a2a4a;
            transition:transform .18s ease, box-shadow .18s ease;
            box-shadow: 0 2px 12px rgba(0,0,0,.4);
        " onmouseover="this.style.transform='translateY(-4px)';this.style.boxShadow='0 8px 24px rgba(99,102,241,.35)'"
           onmouseout="this.style.transform='';this.style.boxShadow='0 2px 12px rgba(0,0,0,.4)'">
            <div style="position:relative;">
                <img src="{v['thumbnail_url']}" alt="{title}"
                     style="width:100%; height:118px; object-fit:cover; display:block;"
                     onerror="this.src='https://img.youtube.com/vi/{v['video_id']}/hqdefault.jpg'">
                <span style="
                    position:absolute; bottom:6px; right:6px;
                    background:rgba(0,0,0,.82); color:#fff;
                    font-size:11px; font-weight:600; padding:2px 6px;
                    border-radius:4px; font-family:monospace;
">▶ {ts}</span>
            </div>
            <div style="padding:10px 12px 12px;">
                <div style="font-size:12.5px; font-weight:600; color:#e2e8f0; line-height:1.4; margin-bottom:4px;">{title}</div>
                <div style="font-size:11px; color:#6366f1; font-weight:500;">{ch}</div>
            </div>
        </a>""")

    inner = "\n".join(cards)
    return f"""
    <div style="margin-top:14px; padding:0 2px;">
        <div style="font-size:11px; font-weight:700; text-transform:uppercase; letter-spacing:1.2px; color:#6366f1; margin-bottom:10px;">
            📺 Related Videos — click to jump to the explanation
        </div>
        <div style="display:flex; gap:12px; flex-wrap:wrap;">
            {inner}
        </div>
    </div>
    """

print("✅ Video suggestion helpers defined")


✅ Video suggestion helpers defined


## Step 5 — Test Thumbnail Generation

Run this cell to see the thumbnail HTML rendered **inline in the notebook**.
This lets you verify thumbnails look correct before launching the full UI.


In [7]:
from IPython.display import HTML, display

test_query = "friction coefficient"
videos = get_video_suggestions(test_query, n=4)

print(f"🔍 Query: '{test_query}'")
print(f"📺 Found {len(videos)} unique videos:\n")
for v in videos:
    mins, secs = v['start_time'] // 60, v['start_time'] % 60
    print(f"  • {v['title']}")
    print(f"    Channel: {v['channel']}")
    print(f"    Jump to: {v['watch_url']}  ({mins}:{secs:02d})")
    print()

# Render the card strip inline
html_output = build_thumbnail_html(videos)
display(HTML(f'<div style="background:#0d0d1a; padding:16px; border-radius:12px;">{html_output}</div>'))


🔍 Query: 'friction coefficient'
📺 Found 1 unique videos:

  • Understanding Friction
    Channel: The Efficient Engineer
    Jump to: https://www.youtube.com/watch?v=mx7NYrUXDB0&t=306s  (5:06)



## Step 6 — Chat Function

`chat_with_agent()` is what Gradio calls every time the user sends a message.

It does three things:
1. Invokes the agent with the question + conversation history
2. Runs `get_video_suggestions()` on the question + start of the answer
3. Returns both the text answer and the thumbnail HTML strip


In [8]:
# Per-session history (keyed by session hash in app.py; single dict here in the notebook)
_session_history: List[BaseMessage] = []


def chat_with_agent(
    message: str,
    history: List[Tuple[str, str]],   # Gradio passes this — we ignore it and use _session_history
) -> Tuple[str, str]:
    """
    Returns (answer_text, thumbnail_html).
    Called by Gradio on every user message.
    """
    global _session_history

    try:
        response = agent_executor.invoke({
            "input":        message,
            "chat_history": _session_history,
            "current_date": datetime.now().strftime("%Y-%m-%d"),
        })
        answer = response.get("output", "Sorry, I could not generate a response.")
    except Exception as e:
        answer = f"Error: {e}"

    # Update history
    _session_history.append(HumanMessage(content=message))
    _session_history.append(AIMessage(content=answer))

    # Build thumbnails from query + first 200 chars of answer
    videos     = get_video_suggestions(message + ' ' + answer[:200], n=4)
    thumb_html = build_thumbnail_html(videos)

    return answer, thumb_html


print("✅ chat_with_agent() defined")
print("   → Notebook uses a single global _session_history list.")
print("   → app.py uses a per-session dict keyed by session hash.")


✅ chat_with_agent() defined
   → Notebook uses a single global _session_history list.
   → app.py uses a per-session dict keyed by session hash.


## Step 7 — End-to-End Single Question Test

Run this before launching the UI to confirm everything works.


In [9]:
from IPython.display import HTML, display

_session_history = []   # reset history for clean test

test_q = "What is the coefficient of friction?"
print(f"❓ {test_q}\n")

answer, thumbs = chat_with_agent(test_q, [])

print(f"💬 Answer (first 400 chars):\n{answer[:400]}...\n")
print(f"📺 Thumbnails generated: {'yes' if thumbs else 'no (no videos found)'}\n")

# Render thumbnails inline
if thumbs:
    display(HTML(f'<div style="background:#0d0d1a;padding:16px;border-radius:12px;">{thumbs}</div>'))


❓ What is the coefficient of friction?



> Entering new AgentExecutor chain...

Invoking: `search_transcripts` with `{'query': 'coefficient of friction'}`


[Result 1]
Video: Understanding Friction
Time: 266.88s
Content: . The coefficient of friction mu accounts for the nature of the two contacting surfaces. Let's look at some typical values for different materials. The difficulty with the coefficient of friction is that it depends on much more than just the materials involved, so these are all very approximate values. Surface roughness has a big impact, because it affects the likelihood of mechanical interlocking at asperities. Even machined surfaces with their directional textures can have different friction coefficients in different directions. Then there's the presence of surface oxide layers, cleanliness, temperature, and even heat treatment that can all have an impact on friction
Link: https://youtube.com/watch?v=mx7NYrUXDB0&t=266s

[Result 2]
Video: Understanding Friction
Time:

## Step 8 — Launch Gradio Inside the Notebook

Running the cell below starts the full UI **inside the notebook** (inline preview).

> ⚠️ This uses a simplified `gr.ChatInterface` to keep the notebook readable.
> The production `app.py` uses the full custom-styled `gr.Blocks` layout with
> the dark theme, custom CSS, and session-isolated history.

**To stop:** interrupt the kernel (■ button) or run the `demo.close()` cell below.


In [ ]:
import gradio as gr

def notebook_chat(message, history):
    answer, thumbs = chat_with_agent(message, history)
    return answer

demo_nb = gr.ChatInterface(
    fn=notebook_chat,
    title="⚙️ Engineering RAG Assistant (Notebook Preview)",
    description=(
        "Ask mechanical engineering questions. "
        "Thumbnails appear in the app.py version — this is a text-only preview."
    ),
    examples=[
        ["What is the coefficient of friction?"],
        ["Explain Young's modulus"],
        ["Which videos cover fatigue failure?"],
    ],
)

demo_nb.launch(inline=True, height=600)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.




> Entering new AgentExecutor chain...

Invoking: `search_transcripts` with `{'query': 'viscosity'}`


[Result 1]
Video: Understanding Viscosity
Time: 94.55s
Content: . But close to the wall the velocity changes very suddenly. In this region the slope is large, and so the shear stress is also large. For most fluids, the relationship between the shear stress and the slope of the velocity profile is linear. And the constant of proportionality is what we call fluid viscosity. In engineering it's usually denoted using the Greek letter Mu. We can think of viscosity as the internal friction of a fluid in motion. It has the effect of smoothing out differences in velocity, by increasing the shear stresses in proportion with the velocity gradient. We can express the linear relationship using this equation
Link: https://youtube.com/watch?v=VvDJyhYSJv8&t=94s

[Result 2]
Video: Understanding Viscosity
Time: 232.01s
Content: . And the kinematic viscosity has units of meters squared per second. The

In [11]:
import gradio as gr
print(gr.__version__)

6.9.0


In [13]:
# Run this to shut down the notebook preview server
demo_nb.close()
print('✅ Server closed')


Closing server running on port: 7860
✅ Server closed


## Step 9 — Switch to Production `app.py`

Once you're happy with the notebook preview, close the notebook server and
run the full styled app from the terminal:

```bash
# From the project root (where app.py lives)
python app.py
```

Then open **http://localhost:7860** in your browser.

### What `app.py` adds on top of this notebook

| Feature | Notebook (Step 8) | app.py |
|---|---|---|
| Dark themed UI | ❌ default Gradio | ✅ custom CSS |
| Thumbnail strip | ❌ prints to stdout | ✅ `gr.HTML` panel below chat |
| Per-user memory | ❌ single global list | ✅ session hash dict |
| Example chips | basic | ✅ styled clickable chips |
| Retry / Clear | basic | ✅ styled action buttons |

### Public link (for sharing / demo day)

Change `share=False` to `share=True` in the `demo.launch()` call in `app.py`
and Gradio will give you a temporary public URL valid for 72 hours.
